In [ ]:
from datasets import load_dataset



In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from datasets import load_dataset
from transformers import (
    CamembertTokenizerFast, 
    CamembertForTokenClassification, 
    DataCollatorForTokenClassification,
    TrainingArguments, 
    Trainer
)
import numpy as np
import evaluate

# 1. Chargement du dataset GSD (Universal Dependencies French)
# Autorise explicitement l'utilisation du script de chargement
from datasets import Dataset, DatasetDict
from conllu import parse_incr

def load_conllu_to_hf(filepath):
    # On ouvre le fichier .conllu local
    with open(filepath, "r", encoding="utf-8") as data_file:
        data_list = []
        for tokenlist in parse_incr(data_file):
            tokens = [token["form"] for token in tokenlist]
            upos = [token["upos"] for token in tokenlist]
            data_list.append({"tokens": tokens, "upos": upos})
    return Dataset.from_list(data_list)

# 

# Charger Sequoia (via le projet Universal Dependencies)
dataset = DatasetDict({
    "train": load_conllu_to_hf("fr_sequoia-ud-train.conllu"),
    "validation": load_conllu_to_hf("fr_sequoia-ud-dev.conllu"),
    "test": load_conllu_to_hf("fr_sequoia-ud-test.conllu")
})

# Renommer les colonnes pour qu'elles correspondent à votre script précédent
# Sequoia utilise 'tokens' et 'upos' par défaut dans ce format
print(f"✅ Sequoia chargé ! Splits disponibles : {dataset.keys()}")


# 1. Liste standard des 17 étiquettes Universal POS (UPOS)
label_list = [
    "ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN", 
    "NUM", "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB", "X"
]

# 2. Création des dictionnaires de correspondance
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# 3. Fonction pour transformer les textes ("NOUN", "VERB") en chiffres (7, 15)
def encode_labels(example):
    # On remplace chaque étiquette par son ID. 
    # Si une étiquette est inconnue ou vide (_), on met -100 (sera ignoré par le modèle)
    example["labels"] = [label2id.get(l, -100) for l in example["upos"]]
    return example

# 4. Appliquer la conversion sur tout le dataset
dataset = dataset.map(encode_labels)
print(f"Nombre de phrases pour l'entraînement : {len(dataset['train'])}")
print(f"Nombre de phrases pour la validation : {len(dataset['validation'])}")

# Pour compter le nombre total de mots (tokens) dans l'entraînement :
total_mots = sum([len(x) for x in dataset['train']['tokens']])
print(f"Nombre total de mots à apprendre : {total_mots}")


print(f"✅ Labels encodés. Exemple de correspondance : {id2label[15]} -> 15")
# 2. Chargement du Tokenizer (version Fast obligatoire pour l'alignement)
model_name = "camembert-base"
tokenizer = CamembertTokenizerFast.from_pretrained(model_name)

# 3. Fonction de prétraitement (Alignement des tokens et labels)
def tokenize_and_align_labels(examples):
    # On tokenise les textes
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    
    labels = []
    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Tokens spéciaux ([CLS], [SEP]) -> -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # C'est le début d'un nouveau mot
                # ON CONVERTIT ICI : on cherche l'ID du label (ex: 'DET' -> 5)
                label_name = label[word_idx]
                label_id = label2id.get(label_name, -100) # -100 si inconnu
                label_ids.append(label_id)
            else:
                # C'est un subword (ex: 'ait' dans 'mangeait') -> -100
                label_ids.append(-100)
            previous_word_idx = word_idx
            
        labels.append(label_ids)

    # On crée la colonne 'labels' qui contient uniquement des ENTIERS
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# On relance le map (pensez à redéfinir label2id avant si nécessaire)
tokenized_gsd = dataset.map(tokenize_and_align_labels, batched=True)

# 4. Chargement du modèle pour la classification de tokens
model = CamembertForTokenClassification.from_pretrained(
    model_name, 
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# 5. Métriques avec 'evaluate'
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("Modèle et données GSD prêts pour l'entraînement !")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Map:   0%|          | 0/2231 [00:00<?, ? examples/s]

Map:   0%|          | 0/412 [00:00<?, ? examples/s]

Map:   0%|          | 0/456 [00:00<?, ? examples/s]

Some weights of CamembertForTokenClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modèle et données GSD prêts pour l'entraînement !


In [2]:
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer

# 1. Le collator s'occupe de mettre les phrases à la même longueur (padding)
data_collator = DataCollatorForTokenClassification(tokenizer)

# 2. Configuration de l'entraînement
training_args = TrainingArguments(
    output_dir="./results_camembert_pos_sequoia",
    eval_strategy="epoch",            # Changé ici : 'eval_strategy' au lieu de 'evaluation_strategy'
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir='./logs',
    load_best_model_at_end=True,      # Pour garder le meilleur modèle (pas le dernier)
    metric_for_best_model="accuracy",
    # --- AJOUTEZ CES DEUX LIGNES ---
    logging_steps=10,       # Affiche la loss tous les 10 steps au lieu de 500
    logging_first_step=True # Affiche la loss dès le tout premier step
    # -------------------------------# Basé sur l'accuracy
)


# 3. Initialisation du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_gsd["train"],
    eval_dataset=tokenized_gsd["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics # La fonction qu'on a définie plus haut
) 

# 4. C'EST PARTI !
trainer.train()
# On utilise le split 'test' qui a été tokenisé
results = trainer.evaluate(tokenized_gsd["test"])

print("Résultats sur le jeu de test :", results)

/tmp/ipykernel_26595/920838926.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.789800,1.665484,0.764375,0.696143,0.728665,0.773577
2,1.239700,1.129840,0.952976,0.948991,0.950979,0.957796
3,1.036400,0.935950,0.971775,0.971673,0.971724,0.974797
4,0.889700,0.824221,0.977947,0.978050,0.977998,0.980098
5,0.805800,0.742006,0.979733,0.980245,0.979989,0.981898
6,0.727000,0.677341,0.979419,0.979931,0.979675,0.981698
7,0.664300,0.617949,0.980673,0.981185,0.980929,0.982798
8,0.612200,0.569096,0.982132,0.982440,0.982286,0.983798
9,0.566800,0.529018,0.981718,0.982231,0.981974,0.983598
10,0.522400,0.494678,0.981925,0.982335,0.982130,0.983898


/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: AUX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib/

/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/jolescro/.local/lib

Résultats sur le jeu de test : {'eval_loss': 0.37526074051856995, 'eval_precision': 0.9889560325067722, 'eval_recall': 0.9905040175310446, 'eval_f1': 0.9897294197382827, 'eval_accuracy': 0.9893468737554759, 'eval_runtime': 0.8913, 'eval_samples_per_second': 511.605, 'eval_steps_per_second': 32.536, 'epoch': 20.0}


In [3]:
import shutil
[shutil.rmtree(os.path.join(training_args.output_dir, d)) for d in os.listdir(training_args.output_dir) if d.startswith("checkpoint")]

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]